<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FranQuant/the-ai-engineer/blob/main/capstones/week03_transformers/week03_fomc_surprise.ipynb)

# How Surprising Is the Fed? Per-Document Bits per Character from a From-Scratch Tiny Transformer on FOMC Text

**J. Francisco Salazar (FranQuant)** · 2026-09-25 · The AI Engineer, Week 3 capstone · Pre-registration: `DESIGN.md` (version printed by the setup cell)

**Abstract.** How surprising is each FOMC document to a language model that has read only earlier FOMC text? A tiny decoder-only transformer, built from scratch, is trained on Federal Reserve statements and minutes of meetings that were public by the end of 2019, and scores every later document by its bits per character (BPC). The split is by meeting, so a meeting's statement and minutes stay together: a near period (2020–2021) and a far period (2022 onward). Three hypotheses were pre-registered before any model saw the real split: far documents are more surprising than near ones (H1); minutes are more surprising than the statement of the same meeting (H2); the document ranking survives a switch to a BPE tokenizer (H3). In the confirmatory run H1 is contradicted: there is no evidence that later documents are more surprising. H2 and H3 are supported. The cell after setup prints the estimates and intervals from the confirmatory file.

**Setup.** Clones the repository at `REF` when the modules are not already present, stops without a CUDA GPU (smoke mode excepted), refuses a real run unless this session cloned the code at `REF`, and prints the library versions. `MODE` is the only setting that differs between modes.

In [ ]:
import time

T_START = time.perf_counter()
TIMES = {}  # stage -> seconds, reported in section 12

import dataclasses  # noqa: E402
import json  # noqa: E402
import math  # noqa: E402
import platform  # noqa: E402
import subprocess  # noqa: E402
import sys  # noqa: E402
from datetime import date  # noqa: E402
from pathlib import Path  # noqa: E402

import torch  # noqa: E402

MODE = "real"
REF = "week03-v2"
DESIGN_VERSION = "v0.8"
# MODE: "real" is the only mode that scores real N/F. "rehearsal" (CUDA,
# full frozen settings) and "smoke" (CPU only, tiny settings) replace N and
# F by FAKE splits cut from T meetings; their numbers are not results.
MODES = ("real", "rehearsal", "smoke")
if MODE not in MODES:
    raise ValueError(f"MODE must be one of {MODES}")
FAKE_SPLIT = MODE != "real"
SMOKE = MODE == "smoke"
if torch.cuda.is_available():
    if SMOKE:
        raise RuntimeError("smoke mode is CPU-only; use MODE = 'rehearsal' "
                           "on a GPU")
    DEVICE = torch.device("cuda")
elif SMOKE:
    DEVICE = torch.device("cpu")
else:
    raise RuntimeError("CUDA GPU required: Runtime -> Change runtime type "
                       "-> T4 GPU, then Run all.")

REPO_URL = "https://github.com/FranQuant/the-ai-engineer.git"
SUBDIR = Path("capstones/week03_transformers")
MODULES = ("data.py", "bpe.py", "model.py", "evaluate.py", "ngram.py",
           "train.py", "analysis.py")


def has_modules(p: Path) -> bool:
    return all((p / m).is_file() for m in MODULES)


cwd = Path.cwd()
W3 = next((p.resolve() for p in (cwd, cwd.parent, cwd / SUBDIR,
                                 cwd / "the-ai-engineer" / SUBDIR)
           if has_modules(p)), None)
CODE_SOURCE = "local"  # "clone" when the modules come from the clone below
if W3 is None:
    dest = cwd / "the-ai-engineer"
    if dest.exists():
        raise RuntimeError(f"{dest} exists but lacks {MODULES}; remove it")
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    REPO_URL, str(dest)], check=True)
    W3 = (dest / SUBDIR).resolve()
    if not has_modules(W3):
        raise RuntimeError(f"clone lacks {MODULES} under {SUBDIR}")
    CODE_SOURCE = "clone"


def uncommitted_changes(path: Path):
    """True if `git status --porcelain` lists changes under `path`, False if
    it lists none, None if git or the repository is unavailable."""
    try:
        out = subprocess.run(["git", "-C", str(path), "status",
                              "--porcelain", "--", "."],
                             capture_output=True, text=True)
    except FileNotFoundError:
        return None
    if out.returncode != 0:
        return None
    return bool(out.stdout.strip())


def real_mode_refusal(path: Path, code_source: str, ref: str):
    """Why a real run must not use the code in `path`, or None if it may:
    this session cloned it, and its HEAD is the commit `ref` names."""
    fix = ("Use Runtime → Disconnect and delete runtime, then Run all "
           "again.")
    if code_source != "clone":
        return (f"real mode needs code this session cloned at {ref}, but "
                f"found existing modules in {path}. {fix}")
    head, target = (subprocess.run(["git", "-C", str(path), "rev-parse", r],
                                   capture_output=True, text=True)
                    for r in ("HEAD", f"{ref}^{{commit}}"))
    if head.returncode or target.returncode or head.stdout != target.stdout:
        head, target = (r.stdout.strip() if r.returncode == 0
                        else "unresolved" for r in (head, target))
        return (f"real mode needs HEAD in {path} at {ref}, but HEAD is "
                f"{head} and {ref} is {target}. {fix}")
    return None


UNCOMMITTED_CHANGES = uncommitted_changes(W3)
if MODE == "real":
    refusal = real_mode_refusal(W3, CODE_SOURCE, REF)
    if refusal:
        raise RuntimeError(refusal)
sys.path.insert(0, str(W3))

import analysis  # noqa: E402
import bpe  # noqa: E402
import data  # noqa: E402
import evaluate  # noqa: E402
import model  # noqa: E402
import ngram  # noqa: E402
import train  # noqa: E402

import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402

GIT_COMMIT = subprocess.run(["git", "-C", str(W3), "rev-parse", "HEAD"],
                            capture_output=True, text=True).stdout.strip()
ENV = {"python": platform.python_version(), "torch": torch.__version__,
       "cuda": torch.version.cuda,
       "gpu": (torch.cuda.get_device_name(0) if DEVICE.type == "cuda"
               else f"none ({DEVICE.type})"),
       "ref": REF, "git_commit": GIT_COMMIT, "code_source": CODE_SOURCE,
       "uncommitted_changes": UNCOMMITTED_CHANGES, "mode": MODE,
       "design_version": DESIGN_VERSION}
for k, v in ENV.items():
    print(f"{k:>14}: {v}")
RUN_DIR = W3 / "runs" / MODE  # each mode has its own directory
FIG_DIR = RUN_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
BANNER = ("#" * 72 + f"\n#  {MODE.upper()} RUN: fake N and F cut from T "
          "meetings. NOT A RESULT.\n" + "#" * 72)
if FAKE_SPLIT:
    print("\n" + BANNER)
TIMES["setup"] = time.perf_counter() - T_START

**Numbers behind the abstract.** Loads `runs/confirmatory_results.json`, the first real run (tag `week03-v2-run1`, §9), and prints its confirmatory estimates. Every confirmatory number in this notebook is read from this file; the prose is qualitative.

In [ ]:
# The confirmatory run (DESIGN §9): the first real run, archived after it.
CONF_PATH = W3 / "runs" / "confirmatory_results.json"
if not CONF_PATH.is_file():
    raise FileNotFoundError(
        f"{CONF_PATH} is missing: REF must name a commit that includes the "
        "archived confirmatory run")
CONF = json.loads(CONF_PATH.read_text())
assert CONF["mode"] == "real"
assert CONF["environment"]["ref"] == "week03-v2-run1"
CR = CONF["results"]
CONF_SCORES = {k: [evaluate.DocumentScore(**d) for d in v]
               for k, v in CONF["scores"].items()}


def fmt_estimate(e):
    """Point, 95% CI and label of an Estimate.as_dict() record."""
    label = f" -> {e['label'].upper()}" if e["label"] else ""
    return (f"{e['point']:+.4f}, 95% CI [{e['ci_low']:+.4f}, "
            f"{e['ci_high']:+.4f}]{label}")


env = CONF["environment"]
print(f"Confirmatory run: tag {env['ref']}, commit {env['git_commit'][:7]}, "
      f"{env['gpu']}, torch {env['torch']}")
for key, what in (("H1", "Δ₁ = mean BPC(F) − mean BPC(N)"),
                  ("H2", "Δ₂ = mean [BPC(minutes) − BPC(statement)], "
                         "matched F meetings"),
                  ("H3", "Spearman ρ(char, BPE) on F")):
    print(f"{key}: {what} = {fmt_estimate(CR[key])}")
means = CONF["mean_bpc"]["char"]
print(f"mean char BPC: N {means['N']:.4f} "
      f"({CONF['counts']['N']['documents']} documents), "
      f"F {means['F']:.4f} ({CONF['counts']['F']['documents']} documents)")
assert [CR[k]["label"] for k in ("H1", "H2", "H3")] == [
    "contradicted", "supported", "supported"]  # as stated in the abstract

## 2. Introduction and hypotheses

The FOMC publishes post-meeting statements and, for scheduled meetings, minutes. Their wording changes from one meeting to the next. A language model trained on a committee's past texts gives one operational measure of how new a document's wording is: its per-document surprise, the average number of bits the model needs per character. Low BPC means the text is predictable from earlier FOMC language; high BPC means it departs from it. The measure is textual, not a market measure (section 9).

The model is the measurement instrument, not the contribution: a small decoder-only transformer built from scratch to the handout's specification, trained on text available before a cutoff and never updated after it.

**Pre-registration.** `DESIGN.md` fixed the split, scoring protocol, statistics, labels and compute settings before any model was trained on the real split; amendments made before the real run are in its change log. The confirmatory result is the first real run, executed from tag `week03-v2-run1` and archived as `runs/confirmatory_results.json`. Any later run of this notebook, this one included, is a reproduction (§9).

Splits (§2, section 3): T = train, meetings available by 2019-12-31; N = near, 2020–2021; F = far, from 2022-01-01.

**Confirmatory hypotheses** (§6; char transformer unless stated). Uncertainty: percentile bootstrap over meetings, 2,000 resamples, fixed seed. Labels: *supported* if the 95% CI excludes 0 in the predicted direction; *contradicted* if the point estimate is ≤ 0, whatever the CI; *inconclusive* otherwise.

- **H1 (drift):** Δ₁ = mean BPC(F) − mean BPC(N) > 0. Surprise grows with distance from the cutoff.
- **H2 (genre):** Δ₂ = mean over matched F meetings of [BPC(minutes) − BPC(statement)] > 0. Rationale (§6): statements are expected to be more formulaic.
- **H3 (instrument robustness):** Spearman ρ between char- and BPE-transformer per-document BPC on F ≥ 0.7, labelled by the threshold alone (CI reported). The instruments differ in tokenizer, raw-character context and vocabulary-dependent parameters together (§5); H3 asks whether the surprise ranking survives that change, not what tokenization alone does.

**Exploratory** (§7, no pass/fail):

- **E1:** the five highest-BPC F documents, with dates. No causal or market claim.
- **E2:** Spearman ρ between the char transformer and a character 5-gram (interpolated Witten–Bell) on F. A high ρ means the transformer adds little ranking information beyond local character statistics.

## 3. Data, normalization, split

The corpus, its manifest and the split manifest are checked against hard-coded SHA-256 values (§2, §9); a mismatch stops the notebook. Normalization is §3. The split is the committed meeting-level manifest; the §2 thresholds and the §3 character check are fail-closed (§10).

In [ ]:
t0 = time.perf_counter()
_docs = data.load_documents()  # SHA-256 + length checks; fails closed
split_manifest = data.load_split_manifest()
rules = split_manifest["rules"]
# Document IDs and real body sizes per split, from the manifest (no text).
MANIFEST_IDS = {sp: {e["document_id"] for m in split_manifest["meetings"]
                     if m["split"] == sp for e in m["documents"]}
                for sp in data.SPLITS}
REAL_CHARS = {k: sum(e["body_code_points"] for m in split_manifest["meetings"]
                     if m["split"] in sps for e in m["documents"])
              for k, sps in (("N+F", ("N", "F")), ("F", ("F",)))}

if FAKE_SPLIT:
    # N and F bodies are dropped right after the load and hash checks.
    t_only = [d for d in _docs if d.split == "T"]
    del _docs
    # FAKE split from T meetings only: the last 20% of T meetings as fake
    # F, the 10% before them as fake N, the rest as training T.
    t_meetings = sorted({d.meeting for d in t_only})
    n_fake_f = round(0.2 * len(t_meetings))
    n_fake_n = round(0.1 * len(t_meetings))
    fake_f = set(t_meetings[len(t_meetings) - n_fake_f:])
    fake_n = set(t_meetings[len(t_meetings) - n_fake_f - n_fake_n:
                            len(t_meetings) - n_fake_f])
    docs = [dataclasses.replace(
        d, split="F" if d.meeting in fake_f else
        "N" if d.meeting in fake_n else "T") for d in t_only]
    del t_only
    last_n = date.fromisoformat(max(fake_n))
    first_f = date.fromisoformat(min(fake_f))
    NF_BOUNDARY = last_n + (first_f - last_n) / 2
else:
    docs = [d for d in _docs if d.split in ("T", "N", "F")]
    del _docs
    NF_BOUNDARY = date.fromisoformat(rules["near_end"])
if data.check_characters(docs):  # §3: counts only, no text
    raise RuntimeError("INFEASIBLE (§10): N/F characters absent from T")

train_docs = [d for d in docs if d.split == "T"]
n_docs = [d for d in docs if d.split == "N"]
f_docs = [d for d in docs if d.split == "F"]


def split_counts(ds):
    by_meeting = {}
    for d in ds:
        by_meeting.setdefault(d.meeting, set()).add(d.genre)
    return {"documents": len(ds), "meetings": len(by_meeting),
            "statements": sum(d.genre == "statement" for d in ds),
            "minutes": sum(d.genre == "minutes" for d in ds),
            "matched_meetings": sum(g == {"statement", "minutes"}
                                    for g in by_meeting.values()),
            "body_chars": sum(len(d.body) for d in ds)}


COUNTS = {s: split_counts(ds) for s, ds in
          (("T", train_docs), ("N", n_docs), ("F", f_docs))}
print(f"{'split':>5} " + " ".join(f"{k:>16}" for k in COUNTS["T"]))
for s, c in COUNTS.items():
    print(f"{s:>5} " + " ".join(f"{v:>16,}" for v in c.values()))
print(f"N/F boundary for plots: {NF_BOUNDARY}")

# §2 feasibility thresholds (the real split only; the fake one is smaller)
FEASIBILITY = {"F documents >= 40": COUNTS["F"]["documents"] >= 40,
               "F matched pairs >= 15": COUNTS["F"]["matched_meetings"] >= 15,
               "N documents >= 16": COUNTS["N"]["documents"] >= 16}
if FAKE_SPLIT:
    print("§2 thresholds: not applied to the fake split")
else:
    for k, ok in FEASIBILITY.items():
        print(f"§2 {k}: {'pass' if ok else 'FAIL'}")
    if not all(FEASIBILITY.values()):
        raise RuntimeError("INFEASIBLE (§10): §2 thresholds fail")
print("§3 character check: pass")


def assert_scorable(ds, splits):
    """Documents (or their scores) are in `splits`. On a fake split every
    one is a real T document by manifest ID, never a real N or F one."""
    ds = list(ds)
    assert {d.split for d in ds} <= set(splits), "unexpected split"
    if FAKE_SPLIT:
        ids = {d.id for d in ds}
        assert not ids & (MANIFEST_IDS["N"] | MANIFEST_IDS["F"]), (
            "real N/F document on a fake split")
        assert ids <= MANIFEST_IDS["T"], "fake split holds a non-T document"


TIMES["data"] = time.perf_counter() - t0

## 4. Model and verification checks

The model code is imported from `model.py` (from scratch: scaled dot-product attention with boolean and additive masks, causal mask, self-attention, multi-head attention, FFN, Pre-LN block, sinusoidal positions, `TinyTransformerLM`). The §11 checks run here, on CPU, with visible output.

In [ ]:
t0 = time.perf_counter()
torch.set_printoptions(precision=4, sci_mode=False)
sdpa = model.scaled_dot_product_attention
CHECKS = {}  # every value printed below, saved in results.json

# §4.3 worked example
Q = torch.tensor([[1., 0], [0, 1], [1, 1]])
K = torch.tensor([[1., 0], [1, 1], [0, 1]])
V = torch.tensor([[1., 0], [0, 2], [3, 1]])
Y, S, A = sdpa(Q, K, V)
print("Q K^T =", Q @ K.T, "S = Q K^T / sqrt(2), shifted by the row max =",
      S, "A = softmax(S) =", A, "Y = A V =", Y, sep="\n")
r, e = 1 / math.sqrt(2), math.exp(-1 / math.sqrt(2))
assert torch.equal(Q @ K.T, torch.tensor([[1., 1, 0], [0, 1, 1], [1, 2, 1]]))
assert torch.allclose(S, torch.tensor([[0., 0, -r], [-r, 0, 0], [-r, 0, -r]]),
                      atol=1e-6)
A_ref = torch.tensor([[1, 1, e], [e, 1, 1], [e, 1, e]])
assert torch.allclose(A, A_ref / A_ref.sum(-1, keepdim=True), atol=1e-6)
assert torch.allclose(Y, torch.tensor([[0.994440, 1.0], [1.401112, 1.203336],
                                       [0.993020, 1.255235]]), atol=1e-5)
CHECKS["worked_example"] = {"QKT": (Q @ K.T).tolist(), "S": S.tolist(),
                            "A": A.tolist(), "Y": Y.tolist()}

# Causal rerun: token i attends to tokens <= i only.
Yc, _, Ac = sdpa(Q, K, V, mask=model.make_causal_mask(3))
print("causal A =", Ac, "causal Y =", Yc, sep="\n")
assert torch.equal(Ac[0], torch.tensor([1., 0, 0]))
assert torch.count_nonzero(torch.triu(Ac, diagonal=1)) == 0
Ya, _, _ = sdpa(Q, K, V, mask=model.make_causal_mask(3, additive=True))
assert torch.equal(Ya, Yc)  # boolean and additive masks agree
CHECKS["causal"] = {"A": Ac.tolist(), "Y": Yc.tolist()}

# Fused (PyTorch) vs manual attention
g = torch.Generator().manual_seed(0)
q, k, v = (torch.randn(2, 4, 16, 8, generator=g) for _ in range(3))
for causal in (False, True):
    manual, _, _ = sdpa(q, k, v, mask=model.make_causal_mask(16)
                        if causal else None)
    fused = torch.nn.functional.scaled_dot_product_attention(
        q, k, v, is_causal=causal)
    diff = (manual - fused).abs().max().item()
    print(f"fused vs manual (causal={causal}): max |diff| = {diff:.2e}")
    assert diff < 1e-5
    CHECKS[f"fused_vs_manual_max_diff_causal_{causal}"] = diff

# MHA with H = 1 equals single-head self-attention
torch.manual_seed(3)
sa = model.SelfAttention(d_model=4, causal=True)
mha = model.MultiHeadAttention(d_model=4, num_heads=1, causal=True)
with torch.no_grad():
    mha.proj_qkv.weight.copy_(torch.cat(
        [sa.W_Q.weight, sa.W_K.weight, sa.W_V.weight]))
    mha.proj_out.weight.copy_(torch.eye(4))
x = torch.randn(1, 5, 4)
diff = (mha(x) - sa(x)).abs().max().item()
print(f"MHA(H=1) vs self-attention: max |diff| = {diff:.2e}")
assert diff < 1e-6
CHECKS["mha_h1_vs_self_attention_max_diff"] = diff

# Uniform logits give loss log V
lm = model.TinyTransformerLM(model.ModelConfig(
    vocab_size=87, d_model=16, num_heads=2, num_layers=1, d_ff=32,
    block_size=8, dropout=0.0))
with torch.no_grad():
    lm.tok_emb.weight.zero_()  # tied head: every logit is 0
_, loss = lm(torch.randint(0, 87, (4, 8)), torch.randint(0, 87, (4, 8)))
print(f"uniform logits: loss {loss.item():.6f}, log V {math.log(87):.6f}")
assert abs(loss.item() - math.log(87)) < 1e-6
CHECKS["uniform_logits"] = {"loss": loss.item(), "log_V": math.log(87)}

# Trivial-pattern overfit: ABAB... is learned to near-zero loss
torch.manual_seed(1)
ab = torch.tensor([0, 1] * 200)
lm = model.TinyTransformerLM(model.ModelConfig(
    vocab_size=2, d_model=16, num_heads=2, num_layers=2, d_ff=32,
    block_size=8, dropout=0.0))
opt = torch.optim.Adam(lm.parameters(), lr=3e-3)
for _ in range(300):
    ix = torch.randint(0, len(ab) - 9, (16,))
    xb = torch.stack([ab[i:i + 8] for i in ix])
    yb = torch.stack([ab[i + 1:i + 9] for i in ix])
    _, loss = lm(xb, yb)
    opt.zero_grad()
    loss.backward()
    opt.step()
print(f"AB overfit: loss after 300 steps {loss.item():.4f}")
assert loss.item() < 0.05
CHECKS["ab_overfit_loss_after_300_steps"] = loss.item()
del lm, mha, sa, opt
TIMES["checks"] = time.perf_counter() - t0
print("all §11 checks passed")

## 5. Training

Frozen §8 settings: C1 (d_model 256, 6 layers, 8 heads, d_ff 1024), block 256, batch 64, 3,696 steps per run; AdamW, lr 3e-4 with 200 warm-up steps then cosine to 3e-5; weight decay 0.1; dropout 0.1; clip 1.0; seed 1; fp16 autocast with GradScaler. Tokenizers and the transformers see T only. N is used for the training curves only (10 evaluations × 20 batches × 64 windows), which no decision reads.

In [ ]:
if SMOKE:
    ARCH = {"d_model": 64, "num_layers": 2, "num_heads": 4, "d_ff": 128}
    BLOCK_SIZE, BPE_VOCAB = 128, 500
    CFG = train.TrainConfig(steps=60, batch_size=8, warmup_steps=10,
                            monitor_batches=2, monitor_batch_size=8)
else:
    ARCH, BLOCK_SIZE, BPE_VOCAB = train.ARCH, train.BLOCK_SIZE, train.BPE_VOCAB
    CFG = train.TrainConfig()
_probe = train.make_model(10, BLOCK_SIZE, CFG.seed, **ARCH)
SETTINGS = {"arch": ARCH, "block_size": BLOCK_SIZE, "bpe_vocab": BPE_VOCAB,
            "train": dataclasses.asdict(CFG),
            "precision": "fp16" if DEVICE.type == "cuda" else "fp32",
            "dropout": _probe.blocks[0].ff.net[-1].p,
            "token_embedding_init_std": ARCH["d_model"] ** -0.5,
            "ngram_order": ngram.ORDER}
del _probe
print(json.dumps(SETTINGS, indent=1))

t0 = time.perf_counter()
char_vocab = data.CharVocab.from_documents(train_docs)
TIMES["char_vocab"] = time.perf_counter() - t0
t0 = time.perf_counter()
bpe_tok = bpe.SimpleBPE.from_documents(train_docs, BPE_VOCAB)
TIMES["bpe_fit"] = time.perf_counter() - t0
assert bpe_tok.vocab_size == BPE_VOCAB
TOKENIZERS = {"char": char_vocab, "bpe": bpe_tok}

t0 = time.perf_counter()
assert_scorable(n_docs, ("N",))
samplers, monitors, TOKEN_STATS = {}, {}, {}
for name, tok in TOKENIZERS.items():
    samplers[name] = data.WindowSampler.from_documents(train_docs, tok,
                                                       BLOCK_SIZE)
    monitors[name] = data.NMonitorSampler.from_documents(n_docs, tok,
                                                         BLOCK_SIZE)
    n_tokens = int(samplers[name].data.numel())
    TOKEN_STATS[name] = {
        "vocab_size": tok.vocab_size, "t_tokens_serialized": n_tokens,
        "tokens_per_char": (n_tokens - data.PREFIX_LEN * len(train_docs))
        / COUNTS["T"]["body_chars"],
        "passes_over_t": CFG.steps * CFG.batch_size * BLOCK_SIZE / n_tokens}
TIMES["tokenize"] = time.perf_counter() - t0
for name, s in TOKEN_STATS.items():
    print(f"{name:>4}: vocab {s['vocab_size']:,}, {s['t_tokens_serialized']:,}"
          f" T tokens ({s['tokens_per_char']:.4f}/char), "
          f"{s['passes_over_t']:.2f} passes over T")
print(f"BPE fit: {TIMES['bpe_fit']:.1f} s")

Train the char transformer: fixed step count, final weights kept, no model selection (§2, §8). `run` prints the parameter count, the measured embedding init std, the time and the GradScaler-skipped updates.

In [ ]:
MODELS, TRAINING = {}, {}


def run(name):
    tok = TOKENIZERS[name]
    net = train.make_model(tok.vocab_size, BLOCK_SIZE, CFG.seed, **ARCH)
    n_params = sum(p.numel() for p in net.parameters())
    init_std = net.tok_emb.weight.std().item()
    print(f"--- {name} transformer: {n_params:,} parameters, token "
          f"embedding init std {init_std:.4f}")
    res = train.train(net, samplers[name], monitors[name], CFG, DEVICE,
                      log_every=max(1, CFG.steps // 10))
    MODELS[name] = net.eval()
    TRAINING[name] = {"n_params": n_params,
                      "token_embedding_init_std_measured": init_std,
                      **dataclasses.asdict(res)}
    TIMES[f"train_{name}"] = res.seconds
    tail = res.train_loss[-50:]
    print(f"{name}: {res.steps:,} steps in {res.seconds:.1f} s "
          f"({res.precision}); mean train loss over the last {len(tail)} "
          f"steps {sum(tail) / len(tail):.4f} nats/token; "
          f"{res.skipped_updates} GradScaler-skipped updates (§8: counted, "
          f"not replaced)")


run("char")

Same recipe for the BPE transformer; only the tokenizer, and with it the vocabulary-dependent parameters, differs (§5).

In [ ]:
run("bpe")

Sampling demo from the char model, greedy and at temperature 0.8 (§11). Qualitative; nothing downstream reads it.

In [ ]:
# One short sampling demo from the char model (§11): greedy, then T = 0.8.
t0 = time.perf_counter()
PROMPT = "The Committee"
prompt_ids = torch.tensor([data.serialize(char_vocab, "statement", PROMPT)],
                          device=DEVICE)
gen = torch.Generator(device=DEVICE).manual_seed(CFG.seed)
SAMPLES = {}
for label, kwargs in (("greedy", {"greedy": True}),
                      ("temperature 0.8", {"temperature": 0.8,
                                           "generator": gen})):
    out = MODELS["char"].generate(prompt_ids, 200, **kwargs)
    SAMPLES[label] = char_vocab.decode(out[0, data.PREFIX_LEN:].tolist())
    print(f"[{label}]\n{SAMPLES[label]}\n")
TIMES["sampling"] = time.perf_counter() - t0

## 6. Scoring protocol

§4: each document is scored on its own from `<BOS>` + genre, with sliding windows at stride block_size / 2; every body target is scored exactly once, in fp32, and the §4 assertions run per document. BPC = summed NLL / (body code points × ln 2). The char transformer scores N and F; the BPE transformer and the n-gram (Witten–Bell, order 5, fit on T) score F.

In [ ]:
assert_scorable(n_docs + f_docs, ("N", "F"))
t0 = time.perf_counter()
SCORES = {"char": evaluate.score_documents(MODELS["char"], char_vocab,
                                           n_docs + f_docs, BLOCK_SIZE)}
TIMES["score_char"] = time.perf_counter() - t0
t0 = time.perf_counter()
SCORES["bpe"] = evaluate.score_documents(MODELS["bpe"], bpe_tok, f_docs,
                                         BLOCK_SIZE)
TIMES["score_bpe"] = time.perf_counter() - t0
t0 = time.perf_counter()
ngram_lm = ngram.WittenBellNgram.from_documents(train_docs)
TIMES["ngram_fit"] = time.perf_counter() - t0
t0 = time.perf_counter()
SCORES["ngram"] = ngram_lm.score_documents(f_docs)
TIMES["score_ngram"] = time.perf_counter() - t0

for name, scores in SCORES.items():
    assert_scorable(scores, ("N", "F"))  # DocumentScore has id and split
    assert len({s.id for s in scores}) == len(scores)
if FAKE_SPLIT:
    print(f"{MODE} guard: every scored document is a real T document "
          "(manifest IDs); no real N/F document was scored")

MEAN_BPC = {name: {sp: float(np.mean([s.bpc for s in scores
                                      if s.split == sp]))
                   for sp in ("N", "F") if any(s.split == sp
                                               for s in scores)}
            for name, scores in SCORES.items()}
for name, means in MEAN_BPC.items():
    print(f"{name:>5}: " + ", ".join(
        f"{sp} {len([s for s in SCORES[name] if s.split == sp])} docs, "
        f"mean BPC {m:.4f}" for sp, m in means.items())
        + f"  ({TIMES['score_' + name]:.1f} s)")

# Rehearsal: scoring time projected to the real N+F (char) and F (BPE,
# n-gram) body sizes from the manifest, at the rate measured here.
SCORING_PROJECTION = None
if FAKE_SPLIT:
    fake_chars = {"N+F": COUNTS["N"]["body_chars"] + COUNTS["F"]["body_chars"],
                  "F": COUNTS["F"]["body_chars"]}
    SCORING_PROJECTION = {}
    for name, target in (("char", "N+F"), ("bpe", "F"), ("ngram", "F")):
        rate = TIMES[f"score_{name}"] / fake_chars[target]
        SCORING_PROJECTION[name] = {
            "target": target, "fake_chars": fake_chars[target],
            "real_chars": REAL_CHARS[target], "measured_s":
            TIMES[f"score_{name}"], "projected_s": rate * REAL_CHARS[target]}
        print(f"projected {name} scoring on real {target} "
              f"({REAL_CHARS[target]:,} chars): "
              f"{SCORING_PROJECTION[name]['projected_s']:.1f} s")

## 7. Results

Percentile bootstrap, 2,000 resamples of meetings, fixed seed (§6). Labels: **supported** if the 95% CI excludes 0 in the predicted direction; **contradicted** if the point estimate is ≤ 0; **inconclusive** otherwise. H3 is labelled by ρ ≥ 0.7 alone, CI reported.

The cell below computes this run's statistics with the frozen `analysis.py`. Each subsection then prints the confirmatory value (from the file) next to this run's; the interpretation refers to the confirmatory value. The `assert` lines check that the facts the prose states hold in the confirmatory file.

In [ ]:
t0 = time.perf_counter()
H1 = analysis.h1_drift(SCORES["char"])
H2 = analysis.h2_genre(SCORES["char"])
H3 = analysis.h3_instruments(SCORES["char"], SCORES["bpe"])
E1 = analysis.e1_top(SCORES["char"])
E2 = analysis.e2_ngram(SCORES["char"], SCORES["ngram"])
TIMES["statistics"] = time.perf_counter() - t0
THIS_RUN = {"H1": H1.as_dict(), "H2": H2.as_dict(), "H3": H3.as_dict(),
            "E2": E2.as_dict()}
CONF_LABEL = f"confirmatory ({CONF['environment']['ref']})"
RUN_LABEL = (f"this run ({MODE}, FAKE split)" if FAKE_SPLIT
             else f"this run ({MODE}, {GIT_COMMIT[:7]})")
RUNS = ((CONF_LABEL, CONF), (RUN_LABEL, None))
NOTE = ("Differences between runs are expected (fp16 GPU training is not "
        "bit-reproducible); the confirmatory run is the result (DESIGN §9).")


def compare(key, what):
    """Print one statistic for the confirmatory run and for this run."""
    print(f"{key}: {what}")
    for label, res in ((CONF_LABEL, CR), (RUN_LABEL, THIS_RUN)):
        e = res[key]
        print(f"  {label:<34} {fmt_estimate(e)}  "
              f"({e['n_documents']} documents, {e['n_meetings']} meetings)")
    print(NOTE)
    if FAKE_SPLIT:
        print(f"This run is {MODE.upper()} on a fake split: NOT A RESULT.")


print(f"statistics computed in {TIMES['statistics']:.1f} s")

Four figures, saved under `runs/<MODE>/figures/`: (1) training curves, T training loss and N monitoring (plot only, §2); (2) char BPC per document over time, N and F (H1, E1); (3) matched F meetings, statement vs minutes (H2); (4) char vs BPE BPC on F (H3). They show this run.

In [ ]:
t0 = time.perf_counter()
# Reference palette, slots 1-2 (validated pair); text in ink tokens.
C1, C2 = "#2a78d6", "#eb6834"
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({
    "figure.dpi": 110, "axes.edgecolor": INK2, "axes.labelcolor": INK,
    "axes.titlecolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "lines.linewidth": 2, "legend.frameon": False, "font.size": 10})
FIGURES = {}


def save(fig, name):
    if FAKE_SPLIT:
        fig.text(0.5, 0.5, f"{MODE.upper()}: NOT A RESULT", ha="center",
                 va="center", fontsize=28, color=INK2, alpha=0.25,
                 rotation=20)
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, bbox_inches="tight")
    FIGURES[name] = str(path)
    plt.show()


# Figure 1: training curves (T training loss, N monitoring) per instrument
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for ax, name in zip(axes, ("char", "bpe")):
    tr = TRAINING[name]
    loss = np.array(tr["train_loss"])
    w = max(1, len(loss) // 50)
    smooth = np.convolve(loss, np.ones(w) / w, mode="valid")
    ax.plot(np.arange(w, len(loss) + 1), smooth, color=C1,
            label=f"T training loss ({w}-step mean)")
    ax.plot([m["step"] for m in tr["monitor"]],
            [m["loss"] for m in tr["monitor"]], color=C2, marker="o",
            markersize=6, label="N monitoring (plot only)")
    ax.set(title=f"{name} transformer", xlabel="step",
           ylabel=f"loss (nats per {name} token)")
axes[0].legend()
save(fig, "fig1_training_curves")

# Figure 2: per-document char BPC over time, N and F
fig, ax = plt.subplots(figsize=(10, 3.8))
for sp, color in (("N", C1), ("F", C2)):
    for genre, marker in (("statement", "o"), ("minutes", "s")):
        pts = [(date.fromisoformat(s.meeting), s.bpc) for s in SCORES["char"]
               if s.split == sp and s.genre == genre]
        if pts:
            ax.scatter(*zip(*pts), color=color, marker=marker, s=36,
                       edgecolors="white", linewidths=0.8,
                       label=f"{sp} {genre}")
ax.axvline(NF_BOUNDARY, color=INK2, linestyle="--", linewidth=1)
ax.annotate("N | F boundary", (NF_BOUNDARY, 1), xycoords=("data",
            "axes fraction"), xytext=(4, -12), textcoords="offset points",
            color=INK2, fontsize=9)
ax.set(title="Char transformer BPC per document", xlabel="meeting date",
       ylabel="bits per character")
ax.legend(ncol=4, loc="upper left", bbox_to_anchor=(0, -0.18))
save(fig, "fig2_bpc_over_time")

# Figure 3: paired genre plot over matched F meetings (H2)
pairs, _ = analysis.genre_pairs(SCORES["char"])
fig, ax = plt.subplots(figsize=(4.5, 4))
for _, m_bpc, s_bpc in pairs:
    ax.plot([0, 1], [s_bpc, m_bpc], color=GRID, linewidth=1, zorder=1)
ax.scatter([0] * len(pairs), [s for _, _, s in pairs], color=C1, s=36,
           zorder=2, label="statement")
ax.scatter([1] * len(pairs), [m for _, m, _ in pairs], color=C2, s=36,
           zorder=2, label="minutes")
ax.set(xticks=[0, 1], xticklabels=["statement", "minutes"], xlim=(-0.4, 1.4),
       title=f"Matched F meetings (n = {len(pairs)})",
       ylabel="char BPC")
ax.legend(loc="upper center")
save(fig, "fig3_genre_pairs")

# Figure 4: char vs BPE BPC on F (H3)
bpe_by_id = {s.id: s.bpc for s in SCORES["bpe"]}
fig, ax = plt.subplots(figsize=(4.8, 4.2))
for genre, color in (("statement", C1), ("minutes", C2)):
    pts = [(s.bpc, bpe_by_id[s.id]) for s in SCORES["char"]
           if s.split == "F" and s.genre == genre]
    ax.scatter(*zip(*pts), color=color, s=36, edgecolors="white",
               linewidths=0.8, label=genre)
ax.set(title=f"F documents: Spearman rho = {H3.point:.3f}",
       xlabel="char transformer BPC", ylabel="BPE transformer BPC")
ax.legend()
save(fig, "fig4_char_vs_bpe")
TIMES["figures"] = time.perf_counter() - t0

### 7.1 H1: drift

Δ₁ with its meeting-bootstrap CI and label, then mean char BPC on N and F, for both runs.

In [ ]:
compare("H1", "Δ₁ = mean BPC(F) − mean BPC(N), char transformer")
for label, means in ((CONF_LABEL, CONF["mean_bpc"]["char"]),
                     (RUN_LABEL, MEAN_BPC["char"])):
    print(f"  {label:<34} mean BPC N {means['N']:.4f}, F {means['F']:.4f}")
h1 = CR["H1"]
assert h1["point"] <= 0 and h1["label"] == "contradicted"
assert h1["ci_low"] < 0 < h1["ci_high"]  # the CI includes 0

**Interpretation.** The §6 rule labels a point estimate ≤ 0 *contradicted*, whatever the CI. The confirmatory Δ₁ is negative: far documents are, on average, no more surprising than near ones, so H1 is contradicted. The CI includes 0, so the data are also compatible with a small drift of either sign; they give no evidence that surprise grows with distance from the cutoff.

§14 anticipated one reason before any score existed: N contains 2020 pandemic-era language, which could raise mean BPC(N). Section 8 (post-hoc) finds the 2020 minutes are the most surprising meeting-year × genre group in the confirmatory run. That is consistent with the anticipated effect, but it does not change the result: the pre-registered test is Δ₁ on the frozen split, and it is contradicted.

### 7.2 H2: genre

Δ₂ over matched F meetings, then the effect in two readable forms: Δ₂ as a share of mean statement BPC, and the share of matched meetings whose minutes score higher than their statement.

In [ ]:
compare("H2", "Δ₂ = mean over matched F meetings of "
              "[BPC(minutes) − BPC(statement)]")
for label, scores in ((CONF_LABEL, CONF_SCORES["char"]),
                      (RUN_LABEL, SCORES["char"])):
    pairs, excluded = analysis.genre_pairs(scores)
    m_mean = float(np.mean([m for _, m, _ in pairs]))
    s_mean = float(np.mean([s for _, _, s in pairs]))
    higher = sum(m > s for _, m, s in pairs)
    print(f"  {label:<34} minutes {m_mean:.4f}, statements {s_mean:.4f} "
          f"(Δ₂ = {(m_mean - s_mean) / s_mean:+.1%} of statement BPC); "
          f"minutes higher in {higher} of {len(pairs)} meetings; "
          f"{excluded} unmatched excluded")
h2 = CR["H2"]
assert h2["label"] == "supported" and h2["ci_low"] > 0

**Interpretation.** Minutes are more surprising than the statement of the same meeting. The whole confirmatory CI lies above 0, so H2 is *supported*. The size is moderate in relative terms (the printed share of statement BPC), and minutes score higher in the printed number of matched meetings. The estimand is body surprise conditional on genre (§3). Statements are typically shorter and more formulaic than minutes and often carry over wording from the previous statement; these are possible explanations that this design neither measures nor separates. It shows that the genre difference exists and has this size, not which property of the text causes it.

### 7.3 H3: instrument robustness

Spearman ρ between char- and BPE-transformer BPC on F, with its CI, then both instruments' mean BPC on F.

In [ ]:
compare("H3", f"Spearman ρ(char, BPE) on F, threshold "
              f"{analysis.H3_THRESHOLD}")
for label, means in ((CONF_LABEL, CONF["mean_bpc"]),
                     (RUN_LABEL, MEAN_BPC)):
    print(f"  {label:<34} mean BPC on F: char {means['char']['F']:.4f}, "
          f"BPE {means['bpe']['F']:.4f}")
h3 = CR["H3"]
assert h3["label"] == "supported"
assert h3["ci_low"] >= analysis.H3_THRESHOLD  # CI also above threshold
assert CONF["mean_bpc"]["bpe"]["F"] < CONF["mean_bpc"]["char"]["F"]

**Interpretation.** ρ is above the pre-registered threshold, so H3 is *supported*; the label uses the threshold alone. In the confirmatory run the lower end of the CI is also above the threshold, so the conclusion does not depend on the point estimate. The two instruments agree on which F documents are surprising, even though they differ in tokenizer, raw-character context and vocabulary-dependent parameters (§5). They do not agree in level: the BPE transformer has the lower mean BPC. H3 compares rankings only, so that gap does not bear on it, and it cannot be attributed to tokenization alone given the confounds.

### 7.4 Exploratory: E1 and E2

E1 lists the five F documents with the highest char BPC in each run. E2 is Spearman ρ between the char transformer and the character 5-gram on F. No labels (§7).

In [ ]:
print("E1: top five F documents by char BPC")
for label, top in ((CONF_LABEL, CR["E1"]), (RUN_LABEL, E1)):
    print(f"  {label}")
    for i, d in enumerate(top, 1):
        print(f"    {i}. {d['meeting']} {d['genre']:<9} {d['bpc']:.4f}")
compare("E2", "Spearman ρ(char transformer, 5-gram) on F")
for label, means in ((CONF_LABEL, CONF["mean_bpc"]),
                     (RUN_LABEL, MEAN_BPC)):
    print(f"  {label:<34} mean BPC on F: char {means['char']['F']:.4f}, "
          f"5-gram {means['ngram']['F']:.4f}")
assert all(d["genre"] == "minutes" for d in CR["E1"])
assert CR["E2"]["point"] < CR["H3"]["point"]
assert CONF["mean_bpc"]["char"]["F"] < CONF["mean_bpc"]["ngram"]["F"]

**Interpretation (exploratory).** E1: in the confirmatory run all five most surprising F documents are minutes, which is H2 seen from the top of the ranking; the dates are printed, and no causal or market claim is made about them. E2: the transformer predicts far better than the 5-gram in level (mean BPC above), but the two rank F documents similarly. ρ(char, 5-gram) is lower than ρ(char, BPE) of H3, yet high: by the §7 reading, much of which document is surprising is already visible to a model that sees only the previous four characters, and the transformer adds limited ranking information beyond local statistics.

## 8. Post-hoc exploration (not pre-registered)

**Not pre-registered.** These three tables were chosen after the confirmatory results had been seen, to describe where the H1 contradiction comes from. They carry no test, no interval and no label, and nothing here changes a §6 result. Each is computed from this run's per-document char scores, then from the confirmatory file:

- (i) mean char BPC by split × genre;
- (ii) mean char BPC by meeting year × genre;
- (iii) the six statements with the highest char BPC (N and F), with meeting dates.

In [ ]:
def mean_table(scores, key):
    """Mean char BPC and document count per (key(score), genre)."""
    groups = {}
    for s in scores:
        groups.setdefault((key(s), s.genre), []).append(s.bpc)
    return {k: (float(np.mean(v)), len(v)) for k, v in sorted(groups.items())}


def print_table(title, table):
    print(title)
    print(f"  {'':>6} {'statement':>16} {'minutes':>16}")
    for row in sorted({r for r, _ in table}):
        cells = (f"{table[row, g][0]:.4f} (n={table[row, g][1]:>2})"
                 if (row, g) in table else "-"
                 for g in ("statement", "minutes"))
        print(f"  {row:>6} " + " ".join(f"{c:>16}" for c in cells))


POSTHOC = {}
for label, scores in ((RUN_LABEL, SCORES["char"]),
                      (CONF_LABEL, CONF_SCORES["char"])):
    by_split = mean_table(scores, lambda s: s.split)
    by_year = mean_table(scores, lambda s: s.meeting[:4])
    top = sorted((s for s in scores if s.genre == "statement"),
                 key=lambda s: -s.bpc)[:6]
    POSTHOC[label] = {"by_split": by_split, "by_year": by_year, "top": top}
    print(f"===== {label}")
    print_table("(i) mean char BPC by split x genre", by_split)
    print_table("(ii) mean char BPC by meeting year x genre", by_year)
    print("(iii) six highest-BPC statements")
    for i, s in enumerate(top, 1):
        print(f"  {i}. {s.meeting} ({s.split}) {s.bpc:.4f}")
    print()
if FAKE_SPLIT:
    print(f"This run is {MODE.upper()} on a fake split: NOT A RESULT.")

c = POSTHOC[CONF_LABEL]
assert max(c["by_year"], key=lambda k: c["by_year"][k][0]) == (
    "2020", "minutes")
assert c["by_split"]["N", "minutes"][0] > c["by_split"]["F", "minutes"][0]
assert c["by_split"]["N", "statement"][0] < c["by_split"]["F", "statement"][0]
assert sum(s.meeting.startswith("2022") for s in c["top"]) > len(c["top"]) / 2

**Reading (post-hoc, descriptive).** In the confirmatory run the two genres move in opposite directions between N and F: F statements score higher than N statements, while F minutes score lower than N minutes (table i). The negative Δ₁ is therefore carried by the minutes. Table (ii) shows the 2020 minutes as the most surprising meeting-year × genre group, consistent with the §14 expectation that pandemic-era language raises N. Table (iii): most of the six most surprising statements come from 2022 meetings, consistent with 2022 statement wording being further from the T-period statements than that of other years. These patterns describe the scores; they do not establish why the wording changed, and they say nothing about markets.

## 9. Limitations and future work

Stated before any result, in DESIGN §14:

- Textual surprise is not market surprise.
- One cutoff, not a rolling backtest.
- N includes 2020 pandemic-era language, which may raise N and make H1 harder to support (sections 7.1 and 8).
- Small models; absolute BPC is not comparable to published language models.
- Tokenizer confounds (§5): BPE sees more raw characters per block and has more vocabulary-dependent parameters.
- Unpinned Colab libraries (§9): results are reproducible to the printed library versions, not bit-identical across Colab updates.
- BPE makes many more passes over its T tokens than char (section 5 prints both); with fixed steps and no early stopping it may overfit, which the N curve in Figure 1 would show. H3 compares rankings, not absolute BPC.

The cell below prints the genre mix of N and F and the confirmatory runtime, which the added limitations use.

In [ ]:
for label, counts in ((CONF_LABEL, CONF["counts"]), (RUN_LABEL, COUNTS)):
    print(label)
    for sp in ("N", "F"):
        c = counts[sp]
        print(f"  {sp}: {c['statements']} statements, {c['minutes']} "
              f"minutes, {c['meetings']} meetings; statement share "
              f"{c['statements'] / c['documents']:.1%}")
TARGET_S, CPF_LIMIT_S = 12 * 60, 15 * 60  # DESIGN §8
conf_s = CONF["total_runtime_s"]
print(f"confirmatory runtime: {conf_s:.1f} s ({conf_s / 60:.1f} min) on "
      f"{CONF['environment']['gpu']}; §8 target {TARGET_S / 60:.0f} min, "
      f"CPF limit {CPF_LIMIT_S / 60:.0f} min")
share = {sp: CONF["counts"][sp]["statements"] / CONF["counts"][sp]["documents"]
         for sp in ("N", "F")}
assert share["N"] > share["F"]
assert TARGET_S < conf_s < CPF_LIMIT_S

**Added limitations.**

- **Genre mix.** N has a larger share of statements than F (printed above; some 2020 unscheduled meetings issued a statement but no minutes). Statements score lower than minutes (H2), so the extra statements lower mean BPC(N) and push Δ₁ upward. The imbalance favours H1; it cannot explain the contradiction.
- **One seed.** The §8 rule left no budget for a second char seed, so every claim rests on seed 1. The bootstrap resamples meetings, not training runs: training variance is not in the intervals.
- **Runtime.** The confirmatory Run all took the time printed above on a T4: over the §8 target of 12 minutes, under the CPF limit of 15. The §8 contingency applies only above 15 minutes and was not triggered.

**Future work.**

- Rolling cutoffs: retrain at successive year boundaries, so drift is measured at several distances from the cutoff instead of one.
- Market-reaction linkage: relate per-document BPC to market moves around each release; only that step could connect textual surprise to market surprise.
- More seeds, to put training variance into the intervals.

## 10. References and reused code

**Reused code**

- Y. Hilpisch, *Attention Mechanisms and Tiny Transformers*, The AI Engineer (TAE) handout. `model.py` follows the handout's skeletons for scaled dot-product attention and the causal mask (`scaled_dot_product_attention`, `make_causal_mask`), `SelfAttention`, `MultiHeadAttention`, sinusoidal `PositionalEncoding`, and the Pre-LN `TransformerBlock` with its `FeedForward`; `TinyTransformerLM` assembles them. The worked example and checks in section 4 follow the handout's §4.3. The `model.py` docstring lists the changes.
- yhilpisch/yoctoGPT (GitHub, https://github.com/yhilpisch/yoctoGPT): reference implementation consulted for the model structure (`model.py`) and the training loop (`train.py`).
- Week 3 v1 of this capstone (tag `week03-v1`): `model.py`, `bpe.py` and `evaluate.py` are ported from it; each docstring lists the changes.

**Literature**

- Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., Kaiser, Ł., & Polosukhin, I. (2017). Attention is all you need. *Advances in Neural Information Processing Systems 30*.
- Sennrich, R., Haddow, B., & Birch, A. (2016). Neural machine translation of rare words with subword units. *Proceedings of the 54th Annual Meeting of the Association for Computational Linguistics*, 1715–1725.
- Witten, I. H., & Bell, T. C. (1991). The zero-frequency problem: Estimating the probabilities of novel events in adaptive text compression. *IEEE Transactions on Information Theory*, 37(4), 1085–1094.
- Efron, B., & Tibshirani, R. J. (1993). *An Introduction to the Bootstrap*. Chapman & Hall.

**Data**

- Board of Governors of the Federal Reserve System, FOMC statements and minutes, federalreserve.gov (https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm). Collected by `build_fomc_corpus.py`, `build_fomc_minutes_corpus.py` and `merge_fomc_corpus.py`; the snapshot is frozen by SHA-256 (section 3).

## 11. Use of AI tools

- **Claude** (claude.ai chat, Anthropic): co-developed the research direction and the pre-registration (`DESIGN.md`) with the author, reviewed audit and review findings against the handout and the CPF rules, and drafted prompts.
- **Claude Code** (Anthropic, Claude Opus 5.5): wrote the Python modules, tests, notebooks and this prose under the author's direction, and ran the local tests and smoke runs.
- **Codex** (OpenAI, via the herdr terminal multiplexer): read-only adversarial reviews of the design and the code. Every finding was triaged and re-verified before any change.
- **The author** chose the direction and made all design decisions and amendments, approved or vetoed every change, ran all Colab executions (pilot, rehearsal, confirmatory run), and made every git commit.
- Earlier v1 commits in this folder carry `Co-Authored-By` trailers for Claude Sonnet 4.6.

The author is responsible for all content.

## 12. Runtime

`runs/<MODE>/results.json` holds every number this run reports above, the post-hoc tables included. The total wall-clock time of this Run all is measured after that file is written, then added to it.

In [ ]:
RESULTS = {
    "design_version": DESIGN_VERSION, "mode": MODE, "environment": ENV,
    "settings": SETTINGS, "checks": CHECKS, "counts": COUNTS,
    "feasibility": None if FAKE_SPLIT else FEASIBILITY,
    "nf_boundary": str(NF_BOUNDARY), "tokenizers": TOKEN_STATS,
    "training": TRAINING,
    "skipped_updates": {k: v["skipped_updates"] for k, v in TRAINING.items()},
    "samples": {"prompt": PROMPT, **SAMPLES}, "mean_bpc": MEAN_BPC,
    "scores": {k: [dataclasses.asdict(s) for s in v]
               for k, v in SCORES.items()},
    "results": {"H1": H1.as_dict(), "H2": H2.as_dict(), "H3": H3.as_dict(),
                "E1": E1, "E2": E2.as_dict(),
                "bootstrap": {"resamples": analysis.RESAMPLES,
                              "seed": analysis.SEED,
                              "ci_level": analysis.CI_LEVEL}},
    "scoring_projection": SCORING_PROJECTION,
    "posthoc": {
        k: [dataclasses.asdict(s) for s in v] if k == "top" else
        {"|".join(g): {"mean_bpc": m, "n": n} for g, (m, n) in v.items()}
        for k, v in POSTHOC[RUN_LABEL].items()},
    "figures": FIGURES, "times_s": TIMES,
}
RESULTS_PATH = RUN_DIR / "results.json"
RESULTS_PATH.write_text(json.dumps(RESULTS, indent=1) + "\n")
TOTAL_S = time.perf_counter() - T_START  # after the full write
RESULTS["total_runtime_s"] = TOTAL_S
if SCORING_PROJECTION is not None:
    # Measured total with fake-split scoring swapped for the projection.
    # Fits on the smaller fake T (BPE, n-gram, tokenizing) are not rescaled.
    RESULTS["projected_real_total_s"] = TOTAL_S + sum(
        v["projected_s"] - v["measured_s"]
        for v in SCORING_PROJECTION.values())
RESULTS_PATH.write_text(json.dumps(RESULTS, indent=1) + "\n")
for k, v in TIMES.items():
    print(f"{k:>12}: {v:7.1f} s")
print(f"saved {RESULTS_PATH}")
print(f"TOTAL RUNTIME: {TOTAL_S:.1f} s ({TOTAL_S / 60:.2f} min)")
if "projected_real_total_s" in RESULTS:
    print(f"projected real-run total (scoring rescaled to real N/F sizes): "
          f"{RESULTS['projected_real_total_s']:.1f} s "
          f"({RESULTS['projected_real_total_s'] / 60:.2f} min)")
if FAKE_SPLIT:
    print(BANNER)